In [24]:
from langgraph.graph import StateGraph,START,END
from dotenv import load_dotenv
from typing import TypedDict, Annotated
from langchain_groq import ChatGroq
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_core.tools import tool
from langgraph.prebuilt import tools_condition,ToolNode
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage
from langgraph.graph.message import add_messages


load_dotenv()




True

In [9]:
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

In [11]:
class researchState(TypedDict):
    messages : Annotated[list[BaseMessage], add_messages]


In [25]:
# tools  
search_tool = DuckDuckGoSearchRun(region = 'us-en')

tools = [search_tool]

llm_with_tools = llm.bind_tools(tools=tools)

 

In [18]:
# nodes

def chat_node(state: researchState) -> researchState:
     
    messages = state['messages']
    response = llm.invoke(messages)
    return {'messages': response}

tool_node = ToolNode(tools)

In [ ]:
graph = StateGraph(researchState)

graph.add_node('chat_node', chat_node)
graph.add_node('tools', tool_node)


graph.add_edge(START, 'chat_node')
graph.add_conditional_edges('chat_node', tools_condition)
graph.add_edge('tools', 'chat_node')

chatbot = graph.compile()


In [27]:
output = chatbot.invoke({'messages': [HumanMessage(content= "what is today's date ?")]})
print(output)

{'messages': [HumanMessage(content="what is today's date ?", additional_kwargs={}, response_metadata={}, id='5bb4b4fb-5db1-4d9f-b2a5-ef333907af36'), AIMessage(content="Today's date is March 22, 2026.", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 13, 'prompt_tokens': 41, 'total_tokens': 54, 'completion_time': 0.038347588, 'completion_tokens_details': None, 'prompt_time': 0.001990307, 'prompt_tokens_details': None, 'queue_time': 0.048980273, 'total_time': 0.040337895}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_68f543a7cc', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d1541-bec8-7ba1-b517-a8fc14b93400-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 41, 'output_tokens': 13, 'total_tokens': 54})]}
